In [ ]:
import pandas as pd
import polars as pl
import ast
import networkx as nx
from network_creation import *

### Read old and new organizations datasets

In [43]:
old_folder = 'cleaned organizations/ok organizations/'
new_folder = 'data new'

# old df
old_1 = pd.read_csv(old_folder + 'org_geo_new_1.csv')
old_2 = pd.read_csv(old_folder + 'org_geo_new_2.csv')
old_3 = pd.read_csv(old_folder + 'org_geo_new_3.csv')
old_4 = pd.read_csv(old_folder + 'org_geo_new_4.csv')
old_5 = pd.read_csv(old_folder + 'org_geo_new_5.csv')
old_6 = pd.read_csv(old_folder + 'org_geo_new_6.csv')
old_7 = pd.read_csv(old_folder + 'org_geo_7.csv')
old_8 = pd.read_csv(old_folder + 'org_geo_8.csv')
old_9 = pd.read_csv(old_folder + 'org_9_reduced.csv')

old_df = [old_1, old_2, old_3, old_4, old_5, old_6, old_7, old_8, old_9]

# new df
new_1 = pd.read_csv(new_folder + '/cordis-fp1projects-csv/organization.csv', sep=';')
new_2 = pd.read_csv(new_folder + '/cordis-fp2projects-csv/organization.csv', sep=';')
new_3 = pd.read_csv(new_folder + '/cordis-fp3projects-csv/organization.csv', sep=';')
new_4 = pd.read_csv(new_folder + '/cordis-fp4projects-csv/organization.csv', sep=';')
new_5 = pd.read_csv(new_folder + '/cordis-fp5projects-csv/organization.csv', sep=';')
new_6 = pd.read_csv(new_folder + '/cordis-fp6projects-csv/organization_ok.csv', sep=';')
new_7 = pd.read_csv(new_folder + '/cordis-fp7projects-csv/organization.csv', sep=';')
new_8 = pd.read_csv(new_folder + '/cordis-h2020projects-csv/organization.csv', sep=';')
new_9 = pd.read_csv(new_folder + '/cordis-HORIZONprojects-csv/organization.csv', sep=';')

new_df = [new_1, new_2, new_3, new_4, new_5, new_6, new_7, new_8, new_9]

/tmp/ipykernel_2795924/570263204.py:6: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  old_2 = pd.read_csv(old_folder + 'org_geo_new_2.csv')
/tmp/ipykernel_2795924/570263204.py:11: DtypeWarning: Columns (9,21) have mixed types. Specify dtype option on import or set low_memory=False.
  old_7 = pd.read_csv(old_folder + 'org_geo_7.csv')
/tmp/ipykernel_2795924/570263204.py:21: DtypeWarning: Columns (3,7,12,18) have mixed types. Specify dtype option on import or set low_memory=False.
  new_4 = pd.read_csv(new_folder + '/cordis-fp4projects-csv/organization.csv', sep=';')
/tmp/ipykernel_2795924/570263204.py:22: DtypeWarning: Columns (3,7,12,18) have mixed types. Specify dtype option on import or set low_memory=False.
  new_5 = pd.read_csv(new_folder + '/cordis-fp5projects-csv/organization.csv', sep=';')
/tmp/ipykernel_2795924/570263204.py:23: DtypeWarning: Columns (3,12) have mixed types. Specify dtype option on import or set low_memory=F

In [44]:
# compare number of elements in old and new df

d = list(zip(old_df, new_df))

for el in d:
    print(len(el[0]), len(el[1]))

7900 7916
19054 19075
31350 31364
65418 67858
81089 81752
75274 75216
155534 140064
148841 178414
89680 115056


### Map organization with geographical info from old dataset to new dataset

In [45]:
# extract organizations

for i in range(len(old_df) - 1):
    old_df[i] = old_df[i].rename(columns={'projectRcn': 'rcn', 'projectReference': 'projectID', 'organization': 'name', 'name_x': 'name', 'city_y': 'city', 'country_y': 'country', 'country_code': 'country_code_nominatim', 'country equal' : 'match_country_org_nom'})
    old_df[i] = old_df[i][['name', 'address', 'lat', 'lon', 'bbox', 'municipality', 'village', 'town', 'city', 'country', 'county', 'state', 'state_district', 'country_code_nominatim', 'importance', 'granularity', 'city_nominatim', 'match_country_org_nom']]

df_old_tot = pd.concat(old_df, ignore_index=True)
df_old_tot['name'] = df_old_tot['name'].str.lower()

# drop duplicates to have unique names of organizations
df_drop = df_old_tot.drop_duplicates(subset=['name'], keep='first').reset_index(drop=True)
df_drop

,name,address,lat,lon,bbox,municipality,village,town,city,country,...,contactForm,contentUpdateDate,rcn,order,role,ecContribution,netEcContribution,totalCost,endOfParticipation,active
0,trinity college,"{'city': 'Dublin', 'county': 'County Dublin', ...",53.349379,-6.260559,"['53.2987342', '53.4105416', '-6.3870259', '-6...",NaN,NaN,NaN,Dublin,Ireland,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"univ di roma ""la sapienza""","{'city': 'Rome', 'county': 'Roma Capitale', 'I...",41.893320,12.482932,"['41.6556417', '42.1410285', '12.2344669', '12...",NaN,NaN,NaN,Rome,Italy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,univ catholique de louvain,"{'city_district': 'Louvain-la-Neuve', 'town': ...",50.674169,4.613791,"['50.6590954', '50.6894891', '4.5956956', '4.6...",NaN,NaN,Ottignies-Louvain-la-Neuve,NaN,Belgium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rijksuniversiteit gent,"{'city': 'Ghent', 'county': 'Gent', 'state': '...",51.053829,3.725012,"['50.9795422', '51.1888864', '3.5797616', '3.8...",NaN,NaN,NaN,Ghent,Belgium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,inst jaime almera,"{'city': 'Almeria', 'province': 'Almeria', 'IS...",36.841420,-2.462814,"['35.9376398', '37.0000125', '-3.0372135', '-2...",NaN,NaN,NaN,Almeria,Spain,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153651,clariant ag,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tarragona,ES,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-07-24 17:09:30,1988813.0,13.0,associatedPartner,NaN,0.0,0,False,NaN
153652,foresight institute,NaN,NaN,NaN,NaN,NaN,NaN,NaN,San Francisco,US,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-07-24 17:09:30,1988815.0,17.0,associatedPartner,NaN,0.0,0,False,NaN
153653,oriensys srl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Strasbourg,FR,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-07-24 17:09:30,1988816.0,18.0,associatedPartner,NaN,0.0,0,False,NaN
153654,the desca foundation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dover,US,...,https://ec.europa.eu/info/funding-tenders/oppo...,2024-07-24 17:09:30,1988814.0,14.0,associatedPartner,NaN,0.0,0,False,NaN


In [48]:
# merge new df with df of unique organizations with geographical info

merged_list = []

for i in range(len(new_df)):
    new_df[i]['name'] = new_df[i]['name'].str.lower()
    org_merged = pd.merge(new_df[i], df_drop, on='name', how='left')
    #org_merged.to_csv('data new/merged_old_new_1/merged_org_' + str(i+1) + '.csv', index=False)
    merged_list += [org_merged]
    '''org_new['country'] = org_new['country'].str.lower()
    org_new['street'] = org_new['street'].str.lower()
    org_new = org_new.rename(columns={'country': 'country_code_original'})
    org_new = org_new[['rcn', 'projectID', 'name', 'role', 'street', 'postalCode', 'city', 'region', 'country_code_original']]
    new_df[i] = org_new.drop_duplicates().reset_index(drop=True)'''

In [52]:
# extract organization with missing information about country

for i in range(len(merged_list)):
    missing = merged_list[i].drop_duplicates(subset=['name'], keep='first').reset_index(drop=True)
    missing = missing[missing['country_code_nominatim'].isna()]
    #missing.to_csv('data new/merged_old_new_1/missing_org_' + str(i+1) + '.csv', index=False)

In [53]:
# add info from original country code column to country column withcountry found by nominatim

for i in range(len(merged_list)):
    df = merged_list[i]
    df['country_code_original_nominatim'] = df['country_x'].str.lower()
    df['country_code_original_nominatim'] = df['country_code_original_nominatim'] .fillna(df['country_code_nominatim'])
    df['country_code_original_nominatim'] = df['country_code_original_nominatim'].replace({'uk': 'gb', 'el': 'gr'})
    #df.to_csv('data new/merged_old_new_1_country_code/country_code_' + str(i+1) + '.csv', index=False)

##### We remove duplicated columns and rename some

In [ ]:
# read organizations files with country added

org_folder = 'data new/merged_old_new_1_country_code/'

org_1 = pd.read_csv(org_folder + 'country_code_1.csv')
org_2 = pd.read_csv(org_folder + 'country_code_2.csv')
org_3 = pd.read_csv(org_folder + 'country_code_3.csv')
org_4 = pd.read_csv(org_folder + 'country_code_4.csv')
org_5 = pd.read_csv(org_folder + 'country_code_5.csv')
org_6 = pd.read_csv(org_folder + 'country_code_6.csv')
org_7 = pd.read_csv(org_folder + 'country_code_7.csv')
org_8 = pd.read_csv(org_folder + 'country_code_8.csv')
org_9 = pd.read_csv(org_folder + 'country_code_9.csv')


# we do it for each organization df 1, 2, ..., 9 -> we manually change it
# remove duplicated columns
org_9 = org_9.drop(columns=['projectID_y', 'projectAcronym_y', 'organisationID_y',
       'vatNumber_y', 'shortName_y', 'SME_y', 'activityType_y', 'street_y',
       'postCode_y', 'nutsCode_y', 'geolocation_y', 'organizationURL_y',
       'contactForm_y', 'contentUpdateDate_y', 'rcn_y', 'order_y', 'role_y',
       'ecContribution_y', 'netEcContribution_y', 'totalCost_y',
       'endOfParticipation_y', 'active_y'])

# save cleaned datasets
org_1.to_csv('data new/organizations/org_1_updated.csv', index=False)
org_2.to_csv('data new/organizations/org_2_updated.csv', index=False)
org_3.to_csv('data new/organizations/org_3_updated.csv', index=False)
org_4.to_csv('data new/organizations/org_4_updated.csv', index=False)
org_5.to_csv('data new/organizations/org_5_updated.csv', index=False)
org_6.to_csv('data new/organizations/org_6_updated.csv', index=False)
org_7.to_csv('data new/organizations/org_7_updated.csv', index=False)
org_8.to_csv('data new/organizations/org_8_updated.csv', index=False)
org_9.to_csv('data new/organizations/org_9_updated_tot.csv', index=False)

/tmp/ipykernel_2795924/873889291.py:8: DtypeWarning: Columns (3,7,12,18,44,46,47,48,49,50,52,53,54,55,56,59,63) have mixed types. Specify dtype option on import or set low_memory=False.
  org_4 = pd.read_csv(org_folder + 'country_code_4.csv')
/tmp/ipykernel_2795924/873889291.py:9: DtypeWarning: Columns (3,7,12,18,44,46,47,48,49,50,51,52,53,54,55,56,59,62,63) have mixed types. Specify dtype option on import or set low_memory=False.
  org_5 = pd.read_csv(org_folder + 'country_code_5.csv')
/tmp/ipykernel_2795924/873889291.py:10: DtypeWarning: Columns (0,3,12,44,46,47,48,49,50,51,52,53,54,55,56,59,62,63) have mixed types. Specify dtype option on import or set low_memory=False.
  org_6 = pd.read_csv(org_folder + 'country_code_6.csv')
/tmp/ipykernel_2795924/873889291.py:11: DtypeWarning: Columns (17,18,20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  org_7 = pd.read_csv(org_folder + 'country_code_7.csv')
/tmp/ipykernel_2795924/873889291.py:12: DtypeWarning: 

In [75]:
# keep only org-proj for projects until 2025

# read proj horizon europe
proj_h_eu = pd.read_csv('data new/projects/project_9.csv')

# read_organizations horizon europe
org_h_eu = pd.read_csv('data new/organizations/org_9_updated_tot.csv')

# projects in horizon europe until 2025
proj_2025 = list(proj_h_eu['id'])

# extract org-proj only for proj until 2025
org_2025 = org_h_eu[org_h_eu['projectID'].isin(proj_2025)]

# save org until 2025
org_2025.to_csv('data new/organizations/org_9_updated.csv', index=False)

/tmp/ipykernel_2795924/3723894560.py:7: DtypeWarning: Columns (6,17,18,20,21,23,24,26,38,42,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  org_h_eu = pd.read_csv('data new/organizations/org_9_updated_tot.csv')


##### NB: here we checked the column "country_code_original_nominatim" and we manually added correct country codes when mssing -> we have the total found country codes in column "country_code_ok"

### Create edgelists for countries network

In [104]:
# read organizations files with country added

org_folder = 'data new/organizations/'

org_1 = pd.read_csv(org_folder + 'org_1_updated.csv')
org_2 = pd.read_csv(org_folder + 'org_2_updated.csv')
org_3 = pd.read_csv(org_folder + 'org_3_updated.csv')
org_4 = pd.read_csv(org_folder + 'org_4_updated.csv')
org_5 = pd.read_csv(org_folder + 'org_5_updated.csv')
org_6 = pd.read_csv(org_folder + 'org_6_updated.csv')
org_7 = pd.read_csv(org_folder + 'org_7_updated.csv')
org_8 = pd.read_csv(org_folder + 'org_8_updated.csv')
org_9 = pd.read_csv(org_folder + 'org_9_updated.csv')

org_list = [org_1, org_2, org_3, org_4, org_5, org_6, org_7, org_8, org_9]

/tmp/ipykernel_2795924/570191269.py:8: DtypeWarning: Columns (3,8,13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  org_4 = pd.read_csv(org_folder + 'org_4_updated.csv')
/tmp/ipykernel_2795924/570191269.py:9: DtypeWarning: Columns (3,8,13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  org_5 = pd.read_csv(org_folder + 'org_5_updated.csv')
/tmp/ipykernel_2795924/570191269.py:10: DtypeWarning: Columns (3,13) have mixed types. Specify dtype option on import or set low_memory=False.
  org_6 = pd.read_csv(org_folder + 'org_6_updated.csv')
/tmp/ipykernel_2795924/570191269.py:11: DtypeWarning: Columns (18,19,21,23) have mixed types. Specify dtype option on import or set low_memory=False.
  org_7 = pd.read_csv(org_folder + 'org_7_updated.csv')
/tmp/ipykernel_2795924/570191269.py:12: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  org_8 = pd.read_csv(org_folder + 'org_8_update

In [105]:
org_9

,projectID,projectAcronym,organisationID,vatNumber,name,country_code_ok,shortName,SME,activityType,street,...,county,state,state_district,country_code_nominatim,importance,granularity,city_nominatim,match_country_org_nom,index,country_code_original_nominatim
0,101234994,OPTIMALMINE,998804636,NaN,university of british columbia,ca,UBC,False,HES,AGRONOMY ROAD 102-6190,...,Metro Vancouver Regional District,British Columbia,NaN,ca,0.686967950648544,city,vancouver,yes,NaN,ca
1,101234994,OPTIMALMINE,950669162,BG822106269,asarel medet ad,bg,NaN,False,PRC,.,...,NaN,Pazardzhik,Panagiurishte,bg,0.501982567558912,city,panagyurishte,yes,NaN,bg
2,101234994,OPTIMALMINE,999881724,AU63942912684,the university of queensland,au,THE UNIVERSITY OF QUEENSLAND,False,HES,ST LUCIA,...,NaN,Queensland,NaN,au,0.669408227759282,city,brisbane city,yes,NaN,au
3,101234994,OPTIMALMINE,926249509,NaN,mining and metallurgy institute bor ltd,rs,Bor Institute,False,OTH,ZELENI BULEVAR 35,...,Jönköping County,NaN,NaN,se,0.27501,city,bor,yes,NaN,rs
4,101234994,OPTIMALMINE,999985417,GB499672470,university of newcastle upon tyne,gb,UNEW,False,HES,KINGS GATE,...,NaN,England,North of Tyne,gb,0.620469585639904,city,newcastle upon tyne,yes,NaN,gb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111675,101132079,Kaleidos,915420526,ESB67031849,wedo project intelligence made easysl,es,WeDo,True,PRC,C/ TAULAT 113 BAJOS 2 A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8041,es
111676,101132079,Kaleidos,988004074,ESG64513450,fundacio privada parc de recerca uab,es,Parc Recerca UAB,False,OTH,CAMPUS DE LA UNIVERSITAT AUTONOMA DE BCN S/N ...,...,Vallès Occidental,Catalonia,NaN,es,0.511535551158634,city,cerdanyola del vallès,yes,NaN,es
111677,101132079,Kaleidos,999993953,IT01131710376,alma mater studiorum - universita di bologna,it,UNIBO,False,HES,VIA ZAMBONI 33,...,Bologna,Emilia-Romagna,NaN,it,0.680577110926843,city,bologna,yes,NaN,it
111678,101132079,Kaleidos,999986484,ESQ0818002H,universitat autonoma de barcelona,es,UAB,False,HES,EDIF A CAMPUS DE LA UAB BELLATERRA CERDANYOLA V,...,Barcelonès,Catalonia,NaN,es,0.743564996950318,city,barcelona,yes,NaN,es


In [87]:
# filter out only EFTA countries

efta_countries = list(pd.read_csv('participants/efta countries.csv')['country'].map(lambda x: x.lower()))

In [ ]:
# group by projectID and put all the countries into a list

output_dir = 'network efta countries/proj_countries/'

grouped_proj_country = []

count = 1
for org in org_list:

    #extract only efta countries
    org_efta = org[org['country_code_ok'].isin(efta_countries)]
    # group by projectID
    grouped_org = org_efta.groupby('projectID').agg(projectID=pd.NamedAgg(column='projectID', aggfunc='first'), country_list=pd.NamedAgg(column='country_code_ok', aggfunc=list))
    grouped_proj_country += [grouped_org]
    grouped_org.to_csv(output_dir + 'grouped_proj_country_' + str(count) + '.csv', index=False)

    count += 1

In [90]:
grouped_proj_country[7]

,projectID,country_list
projectID,,
115797,115797,"[nl, dk, gb, fi, it, gb, fr, de, be, ch, at, s..."
115842,115842,"[nl, it, it, it, ch, gb, ch, gb, de, se]"
115843,115843,"[it, fi, fr, dk, be, de, dk, gb, fr, gb, fr, i..."
115844,115844,"[de, it, it, cz]"
115847,115847,"[ie, gb, be]"
...,...,...
101039254,101039254,"[gb, be, be]"
101046258,101046258,"[se, de, de]"
101046657,101046657,"[se, de, se, at, se, it, at, fr, at, de, de]"


In [109]:
input_dir = 'network efta countries/proj_countries/'
output_dir_edge = 'network efta countries/edgelist/with self-loops/'
output_dir_edge_weighted = 'network efta countries/edgelist weighted/with self-loops/'

for i in range(9):
     print(i+1)

     df = (
          pl.read_csv(
               input_dir + 'grouped_proj_country_' + str(i+1) + '.csv',
               schema_overrides={
                    "projectID": pl.Utf8,   # keep as string
                    "country_list": pl.Utf8    # load as string first
                    }
                    )
                    .with_columns(
                         pl.col("country_list")
                         .map_elements(lambda x: ast.literal_eval(x) if x is not None else None, return_dtype=pl.List(pl.Utf8))   # safely parse only JSON-looking rows
                         .alias("country_list_list")
                         )
                         )
     print(df)

     edgelist = create_edgelist_undirected(df, 'projectID', 'country_list_list', 'alias_country_list', 'source', 'target', output_dir_edge + 'edgelist_' + str(i+1) + '.csv')
     print(edgelist)
     # normalize pairs by sorting (min, max)
     edgelist_weighted_undirected = edgelist.with_columns([
        pl.min_horizontal('source', 'target').alias('node1'),
        pl.max_horizontal('source', 'target').alias('node2'),
     ])

     # group by normalized edge and count
     edgelist_weighted_undirected_final = (
        edgelist_weighted_undirected.group_by(['node1', 'node2'])
        .len()
        .rename({"len": "weight", 'node1': 'source', 'node2': 'target'})
        .sort("weight", descending=True)
     )

     edgelist_weighted_undirected_final.write_csv(output_dir_edge_weighted + 'edgelist_weighted_' + str(i+1) + '.csv')
     
     #edgelist_weighted_undirected_final = edgelist_weighted_undirected(edgelist, 'source', 'target', 'node1', 'node2')

     #print(edgelist_weighted_undirected_final)


1
shape: (3_267, 3)
┌───────────┬────────────────────────────────┬──────────────────────┐
│ projectID ┆ country_list                   ┆ country_list_list    │
│ ---       ┆ ---                            ┆ ---                  │
│ str       ┆ str                            ┆ list[str]            │
╞═══════════╪════════════════════════════════╪══════════════════════╡
│ 1007      ┆ ['fr', 'it', 'it', 'fr', 'gb'] ┆ ["fr", "it", … "gb"] │
│ 1014-BIS  ┆ ['fr']                         ┆ ["fr"]               │
│ 1016      ┆ ['fr']                         ┆ ["fr"]               │
│ 1017      ┆ ['fr']                         ┆ ["fr"]               │
│ 1019      ┆ ['fr']                         ┆ ["fr"]               │
│ …         ┆ …                              ┆ …                    │
│ ST2*0476  ┆ ['gb', 'nl', 'be']             ┆ ["gb", "nl", "be"]   │
│ ST2*0477  ┆ ['gb', 'gr', 'gb']             ┆ ["gb", "gr", "gb"]   │
│ ST2*0478  ┆ ['ie', 'fr', 'es', 'fr']       ┆ ["ie", "fr", … "fr"] │


### Create countries network in nx

In [111]:
# convert df edgelist into list edgelist

list_edgelist_list = []

for i in range(9):
    df_edgelist = pd.read_csv('network efta countries/edgelist weighted/with self-loops/edgelist_weighted_' + str(i+1) + '.csv')
    list_edgelist = df_edgelist_to_list_edgelist(df_edgelist, 'source', 'target', ['weight'])
    print(list_edgelist)
    list_edgelist_list += [list_edgelist]

[('fr', 'gb', {'weight': 944}), ('de', 'fr', {'weight': 806}), ('fr', 'fr', {'weight': 732}), ('de', 'gb', {'weight': 727}), ('fr', 'it', {'weight': 692}), ('de', 'it', {'weight': 443}), ('gb', 'gb', {'weight': 443}), ('de', 'de', {'weight': 417}), ('gb', 'it', {'weight': 416}), ('gb', 'nl', {'weight': 374}), ('be', 'fr', {'weight': 374}), ('it', 'it', {'weight': 302}), ('fr', 'nl', {'weight': 289}), ('de', 'nl', {'weight': 268}), ('be', 'de', {'weight': 259}), ('be', 'gb', {'weight': 230}), ('dk', 'gb', {'weight': 212}), ('es', 'fr', {'weight': 195}), ('it', 'nl', {'weight': 176}), ('de', 'dk', {'weight': 165}), ('gb', 'ie', {'weight': 156}), ('be', 'it', {'weight': 138}), ('be', 'nl', {'weight': 133}), ('es', 'gb', {'weight': 132}), ('de', 'es', {'weight': 125}), ('fr', 'ie', {'weight': 121}), ('fr', 'gr', {'weight': 121}), ('dk', 'fr', {'weight': 120}), ('be', 'be', {'weight': 117}), ('gb', 'gr', {'weight': 110}), ('de', 'gr', {'weight': 108}), ('nl', 'nl', {'weight': 108}), ('gb', 

In [112]:
# create network

output_dir = 'network efta countries/nx graph/with self-loops/'

for i in range(9):
    print(list_edgelist_list[i])

    edgelist = list_edgelist_list[i]

    # save graphs
    create_network_nx(list_edgelist_list[i], {}, 'undirected', output_dir + 'network_' + str(i+1) + '.graphml')

[('fr', 'gb', {'weight': 944}), ('de', 'fr', {'weight': 806}), ('fr', 'fr', {'weight': 732}), ('de', 'gb', {'weight': 727}), ('fr', 'it', {'weight': 692}), ('de', 'it', {'weight': 443}), ('gb', 'gb', {'weight': 443}), ('de', 'de', {'weight': 417}), ('gb', 'it', {'weight': 416}), ('gb', 'nl', {'weight': 374}), ('be', 'fr', {'weight': 374}), ('it', 'it', {'weight': 302}), ('fr', 'nl', {'weight': 289}), ('de', 'nl', {'weight': 268}), ('be', 'de', {'weight': 259}), ('be', 'gb', {'weight': 230}), ('dk', 'gb', {'weight': 212}), ('es', 'fr', {'weight': 195}), ('it', 'nl', {'weight': 176}), ('de', 'dk', {'weight': 165}), ('gb', 'ie', {'weight': 156}), ('be', 'it', {'weight': 138}), ('be', 'nl', {'weight': 133}), ('es', 'gb', {'weight': 132}), ('de', 'es', {'weight': 125}), ('fr', 'ie', {'weight': 121}), ('fr', 'gr', {'weight': 121}), ('dk', 'fr', {'weight': 120}), ('be', 'be', {'weight': 117}), ('gb', 'gr', {'weight': 110}), ('de', 'gr', {'weight': 108}), ('nl', 'nl', {'weight': 108}), ('gb', 

In [114]:
# create networks without self-loops

input_dir = 'network efta countries/nx graph/with self-loops/'
output_dir = 'network efta countries/nx graph/without self-loops/'

for i in range(1, 10):

    g = nx.read_graphml(input_dir + 'network_' + str(i) + '.graphml')
    g.remove_edges_from(nx.selfloop_edges(g))

    nx.write_graphml(g, output_dir + 'network_' + str(i) + '.graphml')

In [116]:
# create edgelist without self-loops

input_dir = 'network efta countries/nx graph/without self-loops/'
output_dir = 'network efta countries/edgelist/without self-loops/'

for i in range(1, 10):
    
    g = nx.read_graphml(input_dir + 'network_' + str(i) + '.graphml')
    edges = list(g.edges(data=True))
    edge_df = list_edgelist_to_df_edgelist(edges)

    edge_df.to_csv(output_dir + 'edgelist_' + str(i) + '.csv')

In [117]:
# create edgelist with weights without self-loops

input_dir = 'network efta countries/edgelist/without self-loops/'
output_dir = 'network efta countries/edgelist weighted/without self-loops/'

for i in range(1, 10):

    edgelist = pl.read_csv(input_dir + 'edgelist_' + str(i) + '.csv')

    # normalize pairs by sorting (min, max)
    edgelist_weighted_undirected = edgelist.with_columns([
        pl.min_horizontal('source', 'target').alias('node1'),
        pl.max_horizontal('source', 'target').alias('node2'),
    ])

    # group by normalized pair and sum weights
    edgelist_weighted_undirected_final = (
        edgelist_weighted_undirected.group_by(['node1', 'node2'])
                                              .agg(pl.col('weight').sum().alias('total_weight'))
                                              .sort('total_weight', descending=True)
                                              .rename({'node1': 'source', 'node2': 'target', 'total_weight': 'weight'})
    )

    edgelist_weighted_undirected_final.write_csv(output_dir + 'edgelist_weighted_' + str(i) + '.csv')

In [99]:
g = gt.load_graph('network efta countries/nx graph/without self-loops/network_NEW_9.graphml')
for v in g.vertices():
    print(f"Node {int(v)}:")
    for prop_name, prop in g.vp.items():
        print(f"  {prop_name}: {prop[v]}")
print("Edge Properties:")
for prop_name, prop_map in g.ep.items():
    print(f"Property: {prop_name}")
    for e in g.edges():
        print(f"Edge {e}: {prop_name} = {prop_map[e]}")

Node 0:
  _graphml_vertex_id: es
Node 1:
  _graphml_vertex_id: it
Node 2:
  _graphml_vertex_id: de
Node 3:
  _graphml_vertex_id: fr
Node 4:
  _graphml_vertex_id: nl
Node 5:
  _graphml_vertex_id: be
Node 6:
  _graphml_vertex_id: gr
Node 7:
  _graphml_vertex_id: gb
Node 8:
  _graphml_vertex_id: at
Node 9:
  _graphml_vertex_id: pt
Node 10:
  _graphml_vertex_id: ch
Node 11:
  _graphml_vertex_id: se
Node 12:
  _graphml_vertex_id: fi
Node 13:
  _graphml_vertex_id: dk
Node 14:
  _graphml_vertex_id: pl
Node 15:
  _graphml_vertex_id: no
Node 16:
  _graphml_vertex_id: ie
Node 17:
  _graphml_vertex_id: cz
Node 18:
  _graphml_vertex_id: si
Node 19:
  _graphml_vertex_id: ro
Node 20:
  _graphml_vertex_id: cy
Node 21:
  _graphml_vertex_id: hu
Node 22:
  _graphml_vertex_id: ee
Node 23:
  _graphml_vertex_id: bg
Node 24:
  _graphml_vertex_id: hr
Node 25:
  _graphml_vertex_id: lt
Node 26:
  _graphml_vertex_id: sk
Node 27:
  _graphml_vertex_id: lu
Node 28:
  _graphml_vertex_id: lv
Node 29:
  _graphml_vert

In [100]:
g = nx.Graph()
edgelist = [('it', 'fr', {'weight': 2}), ]
# set edgelist
g.add_edges_from(edgelist)